# LSTM 验证集评估

加载验证损失最低的 LSTM 检查点，与 TextCNN 和传统机器学习基线比较。测试集不参与本阶段评估。

In [ ]:
# 加载 LSTM 检查点和验证集
import pandas as pd
import torch
from torch.utils.data import DataLoader

from sentiment_analysis.evaluation import classification_metrics
from sentiment_analysis.lstm import LSTMClassifier
from sentiment_analysis.paths import CHECKPOINTS_DIR, CLEANED_DIR, TABLES_DIR, FIGURES_DIR
from sentiment_analysis.sequence_data import SentimentDataset, TOKEN_PATTERN
from sentiment_analysis.textcnn import TextCNN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(CHECKPOINTS_DIR / "lstm_best.pt", map_location=device, weights_only=True)

if checkpoint["token_pattern"] != TOKEN_PATTERN.pattern or not checkpoint["lowercase"]:
    raise ValueError("Checkpoint tokenizer does not match the current sequence processing.")

validation_data = pd.read_csv(CLEANED_DIR / "val.csv")
validation_dataset = SentimentDataset(
    validation_data, checkpoint["vocabulary"], checkpoint["max_length"],
)
validation_loader = DataLoader(validation_dataset, batch_size=32, shuffle=False)

lstm_model = LSTMClassifier(**checkpoint["model_config"]).to(device)
lstm_model.load_state_dict(checkpoint["model_state_dict"])
lstm_model.eval()

print("Device:", device)
print(f"LSTM checkpoint epoch: {checkpoint['epoch']}")
print(f"Validation samples: {len(validation_dataset):,}")

In [ ]:
# 生成验证集预测
def predict(model, data_loader, device):
    model.eval()
    labels_all = []
    predictions_all = []

    with torch.no_grad():
        for token_ids, lengths, labels in data_loader:
            logits = model(token_ids.to(device), lengths)
            labels_all.extend(labels.tolist())
            predictions_all.extend(logits.argmax(dim=1).cpu().tolist())

    return labels_all, predictions_all

y_true, y_prediction_lstm = predict(lstm_model, validation_loader, device)
print(f"Predictions: {len(y_prediction_lstm):,}")

In [ ]:
# 比较 LSTM、TextCNN 和传统基线的验证集指标
lstm_result = {
    "EXPERIMENT": "LSTM",
    "FEATURES": "Token sequence",
    "MODEL": "LSTM",
    **classification_metrics(y_true, y_prediction_lstm),
}

baseline_comparison = pd.read_csv(TABLES_DIR / "textcnn_validation_comparison.csv")
comparison = pd.concat([baseline_comparison, pd.DataFrame([lstm_result])], ignore_index=True)
display(comparison.round(4))

TABLES_DIR.mkdir(parents=True, exist_ok=True)
comparison.to_csv(TABLES_DIR / "lstm_validation_comparison.csv", index=False)

In [ ]:
# 绘制 LSTM 验证集混淆矩阵
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

matrix = confusion_matrix(y_true, y_prediction_lstm, labels=[0, 1])
figure, axis = plt.subplots(figsize=(6, 5))
sns.heatmap(
    matrix, annot=True, fmt="d", cmap="Blues", ax=axis,
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"],
)
axis.set_xlabel("Predicted label")
axis.set_ylabel("True label")
axis.set_title("LSTM Validation Confusion Matrix")
figure.tight_layout()

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
figure.savefig(FIGURES_DIR / "lstm_validation_confusion_matrix.png", dpi=150)
plt.show()

## CNN 与 LSTM 的预测比较

先统计两种模型在验证集上预测正确和错误的交叉情况，后续再分析具体评论和融合效果。

In [ ]:
# 使用 TextCNN 最佳检查点生成相同验证集的预测
textcnn_checkpoint = torch.load(
    CHECKPOINTS_DIR / "textcnn_extended_best.pt", map_location=device, weights_only=True,
)

if (textcnn_checkpoint["vocabulary"] != checkpoint["vocabulary"]
        or textcnn_checkpoint["max_length"] != checkpoint["max_length"]
        or textcnn_checkpoint["token_pattern"] != checkpoint["token_pattern"]
        or textcnn_checkpoint["lowercase"] != checkpoint["lowercase"]):
    raise ValueError("TextCNN and LSTM sequence processing does not match.")

textcnn_model = TextCNN(**textcnn_checkpoint["model_config"]).to(device)
textcnn_model.load_state_dict(textcnn_checkpoint["model_state_dict"])
textcnn_labels, y_prediction_textcnn = predict(textcnn_model, validation_loader, device)

if textcnn_labels != y_true:
    raise ValueError("Validation labels are not aligned.")

prediction_comparison = validation_data[["text", "label"]].copy()
prediction_comparison["TextCNN"] = y_prediction_textcnn
prediction_comparison["LSTM"] = y_prediction_lstm

textcnn_correct = prediction_comparison["TextCNN"] == prediction_comparison["label"]
lstm_correct = prediction_comparison["LSTM"] == prediction_comparison["label"]

comparison_summary = pd.DataFrame([
    {"CASE": "Both correct", "COUNT": (textcnn_correct & lstm_correct).sum()},
    {"CASE": "Only TextCNN correct", "COUNT": (textcnn_correct & ~lstm_correct).sum()},
    {"CASE": "Only LSTM correct", "COUNT": (~textcnn_correct & lstm_correct).sum()},
    {"CASE": "Both wrong", "COUNT": (~textcnn_correct & ~lstm_correct).sum()},
])

display(comparison_summary)
comparison_summary.to_csv(TABLES_DIR / "textcnn_lstm_prediction_comparison.csv", index=False)

In [ ]:
# 查看两种模型预测分歧和共同错误的评论
print("Only TextCNN correct:")
display(prediction_comparison[textcnn_correct & ~lstm_correct].head(3))

print("Only LSTM correct:")
display(prediction_comparison[~textcnn_correct & lstm_correct].head(3))

print("Both wrong:")
display(prediction_comparison[~textcnn_correct & ~lstm_correct].head(3))

## CNN-LSTM 软投票

对两个模型的正类概率进行加权平均，先比较等权重，再在验证集上搜索权重。概率大于 0.5 时预测为正面；恰好为 0.5 时预测为负面，与两个模型的 argmax 规则一致。验证集同时用于权重选择和结果展示，这里的最佳分数只是探索结果。

In [ ]:
# 获取两个模型的正类概率
def predict_positive_probabilities(model, data_loader, device):
    model.eval()
    labels_all = []
    probabilities_all = []

    with torch.no_grad():
        for token_ids, lengths, labels in data_loader:
            logits = model(token_ids.to(device), lengths)
            probabilities = torch.softmax(logits, dim=1)[:, 1]
            labels_all.extend(labels.tolist())
            probabilities_all.extend(probabilities.cpu().tolist())

    return labels_all, probabilities_all

textcnn_labels, textcnn_probabilities = predict_positive_probabilities(
    textcnn_model, validation_loader, device,
)
lstm_labels, lstm_probabilities = predict_positive_probabilities(
    lstm_model, validation_loader, device,
)

if textcnn_labels != y_true or lstm_labels != y_true:
    raise ValueError("Validation labels are not aligned.")

print(f"TextCNN probabilities: {len(textcnn_probabilities):,}")
print(f"LSTM probabilities: {len(lstm_probabilities):,}")

In [ ]:
# 比较不同权重，CNN 权重从 0 到 1，LSTM 权重为其余部分
import numpy as np

textcnn_probabilities = np.array(textcnn_probabilities)
lstm_probabilities = np.array(lstm_probabilities)

weight_results = []

for step in range(11):
    textcnn_weight = step / 10
    lstm_weight = 1 - textcnn_weight
    ensemble_probabilities = (
        textcnn_weight * textcnn_probabilities + lstm_weight * lstm_probabilities
    )
    ensemble_prediction = (ensemble_probabilities > 0.5).astype(int)

    weight_results.append({
        "TEXTCNN WEIGHT": textcnn_weight,
        "LSTM WEIGHT": lstm_weight,
        **classification_metrics(y_true, ensemble_prediction),
    })

weight_comparison = pd.DataFrame(weight_results)
display(weight_comparison.round(4))
weight_comparison.to_csv(TABLES_DIR / "textcnn_lstm_ensemble_weights.csv", index=False)

In [ ]:
# 按验证集 F1 选择权重；若并列则优先选择更接近等权重的配置
best_candidates = weight_comparison[weight_comparison["F1"] == weight_comparison["F1"].max()].copy()
best_candidates["DISTANCE TO EQUAL"] = (best_candidates["TEXTCNN WEIGHT"] - 0.5).abs()
best_weight_result = best_candidates.sort_values(
    ["DISTANCE TO EQUAL", "TEXTCNN WEIGHT"],
).iloc[0]
best_textcnn_weight = best_weight_result["TEXTCNN WEIGHT"]

equal_result = weight_comparison[weight_comparison["TEXTCNN WEIGHT"] == 0.5].iloc[0]
ensemble_results = pd.DataFrame([
    {"MODEL": "TextCNN", **classification_metrics(y_true, y_prediction_textcnn)},
    {"MODEL": "LSTM", **classification_metrics(y_true, y_prediction_lstm)},
    {"MODEL": "Equal-weight ensemble", **equal_result[["ACCURACY", "PRECISION", "RECALL", "F1"]].to_dict()},
    {"MODEL": "Selected-weight ensemble", **best_weight_result[["ACCURACY", "PRECISION", "RECALL", "F1"]].to_dict()},
])

print(f"Selected weights: TextCNN {best_textcnn_weight:.1f}, LSTM {1 - best_textcnn_weight:.1f}")
display(ensemble_results.round(4))
ensemble_results.to_csv(TABLES_DIR / "textcnn_lstm_ensemble_comparison.csv", index=False)

In [ ]:
# 观察权重与验证集 F1 的关系
figure, axis = plt.subplots(figsize=(8, 5))
axis.plot(weight_comparison["TEXTCNN WEIGHT"], weight_comparison["F1"], marker="o")
axis.set_xlabel("TextCNN weight (LSTM weight = 1 - TextCNN weight)")
axis.set_ylabel("Validation F1")
axis.set_title("Ensemble weight comparison")
axis.set_xticks(weight_comparison["TEXTCNN WEIGHT"])
axis.set_ylim(0, 1)
axis.grid(alpha=0.3)
figure.tight_layout()

figure.savefig(FIGURES_DIR / "textcnn_lstm_ensemble_weights.png", dpi=150)
plt.show()

In [ ]:
# 检查所选权重的融合相对于单模型纠正和引入的错误
selected_probabilities = (
    best_textcnn_weight * textcnn_probabilities + (1 - best_textcnn_weight) * lstm_probabilities
)
prediction_comparison["Ensemble"] = (selected_probabilities > 0.5).astype(int)
ensemble_correct = prediction_comparison["Ensemble"] == prediction_comparison["label"]

error_changes = pd.DataFrame([
    {"MODEL": "TextCNN", "CORRECTED": (~textcnn_correct & ensemble_correct).sum(),
     "NEW ERRORS": (textcnn_correct & ~ensemble_correct).sum()},
    {"MODEL": "LSTM", "CORRECTED": (~lstm_correct & ensemble_correct).sum(),
     "NEW ERRORS": (lstm_correct & ~ensemble_correct).sum()},
])
display(error_changes)
error_changes.to_csv(TABLES_DIR / "textcnn_lstm_ensemble_error_changes.csv", index=False)